In [3]:
import requests
from lakehouse.daft import bronze
import json
import daft
from deltalake.table import TableMerger, DeltaTable


In [4]:
CATALOG = "daft_catalog"

# 1. Set Up and Data

In [5]:
res = []
resource = "planets"
query = f"https://swapi.tech/api/{resource}"
json_request = requests.get(query).json()
res.extend(json_request["results"])

while json_request["next"]:
    json_request = requests.get(json_request["next"]).json()
    res.extend(json_request["results"])

In [6]:
res

[{'uid': '1',
  'name': 'Tatooine',
  'url': 'https://www.swapi.tech/api/planets/1'},
 {'uid': '2',
  'name': 'Alderaan',
  'url': 'https://www.swapi.tech/api/planets/2'},
 {'uid': '3',
  'name': 'Yavin IV',
  'url': 'https://www.swapi.tech/api/planets/3'},
 {'uid': '4', 'name': 'Hoth', 'url': 'https://www.swapi.tech/api/planets/4'},
 {'uid': '5',
  'name': 'Dagobah',
  'url': 'https://www.swapi.tech/api/planets/5'},
 {'uid': '6', 'name': 'Bespin', 'url': 'https://www.swapi.tech/api/planets/6'},
 {'uid': '7', 'name': 'Endor', 'url': 'https://www.swapi.tech/api/planets/7'},
 {'uid': '8', 'name': 'Naboo', 'url': 'https://www.swapi.tech/api/planets/8'},
 {'uid': '9',
  'name': 'Coruscant',
  'url': 'https://www.swapi.tech/api/planets/9'},
 {'uid': '10',
  'name': 'Kamino',
  'url': 'https://www.swapi.tech/api/planets/10'},
 {'uid': '11',
  'name': 'Geonosis',
  'url': 'https://www.swapi.tech/api/planets/11'},
 {'uid': '12',
  'name': 'Utapau',
  'url': 'https://www.swapi.tech/api/planets/

In [7]:
df = daft.from_pylist(res)
df.show()

nameUtf8,uidUtf8,urlUtf8
Tatooine,1,https://www.swapi.tech/api/planets/1
Alderaan,2,https://www.swapi.tech/api/planets/2
Yavin IV,3,https://www.swapi.tech/api/planets/3
Hoth,4,https://www.swapi.tech/api/planets/4
Dagobah,5,https://www.swapi.tech/api/planets/5
Bespin,6,https://www.swapi.tech/api/planets/6
Endor,7,https://www.swapi.tech/api/planets/7
Naboo,8,https://www.swapi.tech/api/planets/8


In [8]:
@daft.udf(return_dtype=daft.DataType.string())
def get_properties(urls: daft.Series) -> list:
    result = []
    for url in urls.to_pylist():
        json_request = requests.get(url).json()
        result.append(json.dumps(json_request["result"]["properties"]))
    return result

In [9]:
df = df.with_column("properties", get_properties(daft.col("url")))

In [10]:
df.show()

nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
Tatooine,1,https://www.swapi.tech/api/planets/1,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""arid"", ""surface_water"": ""1"", ""name"": ""Tatooine"", ""diameter"": ""10465"", ""rotation_period"": ""23"", ""terrain"": ""desert"", ""gravity"": ""1 standard"", ""orbital_period"": ""304"", ""population"": ""200000"", ""url"": ""https://www.swapi.tech/api/planets/1""}"
Alderaan,2,https://www.swapi.tech/api/planets/2,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""40"", ""name"": ""Alderaan"", ""diameter"": ""12500"", ""rotation_period"": ""24"", ""terrain"": ""grasslands, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""364"", ""population"": ""2000000000"", ""url"": ""https://www.swapi.tech/api/planets/2""}"
Yavin IV,3,https://www.swapi.tech/api/planets/3,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate, tropical"", ""surface_water"": ""8"", ""name"": ""Yavin IV"", ""diameter"": ""10200"", ""rotation_period"": ""24"", ""terrain"": ""jungle, rainforests"", ""gravity"": ""1 standard"", ""orbital_period"": ""4818"", ""population"": ""1000"", ""url"": ""https://www.swapi.tech/api/planets/3""}"
Hoth,4,https://www.swapi.tech/api/planets/4,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""frozen"", ""surface_water"": ""100"", ""name"": ""Hoth"", ""diameter"": ""7200"", ""rotation_period"": ""23"", ""terrain"": ""tundra, ice caves, mountain ranges"", ""gravity"": ""1.1 standard"", ""orbital_period"": ""549"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/4""}"
Dagobah,5,https://www.swapi.tech/api/planets/5,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""murky"", ""surface_water"": ""8"", ""name"": ""Dagobah"", ""diameter"": ""8900"", ""rotation_period"": ""23"", ""terrain"": ""swamp, jungles"", ""gravity"": ""N/A"", ""orbital_period"": ""341"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/5""}"
Bespin,6,https://www.swapi.tech/api/planets/6,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""0"", ""name"": ""Bespin"", ""diameter"": ""118000"", ""rotation_period"": ""12"", ""terrain"": ""gas giant"", ""gravity"": ""1.5 (surface), 1 standard (Cloud City)"", ""orbital_period"": ""5110"", ""population"": ""6000000"", ""url"": ""https://www.swapi.tech/api/planets/6""}"
Endor,7,https://www.swapi.tech/api/planets/7,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""8"", ""name"": ""Endor"", ""diameter"": ""4900"", ""rotation_period"": ""18"", ""terrain"": ""forests, mountains, lakes"", ""gravity"": ""0.85 standard"", ""orbital_period"": ""402"", ""population"": ""30000000"", ""url"": ""https://www.swapi.tech/api/planets/7""}"
Naboo,8,https://www.swapi.tech/api/planets/8,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""12"", ""name"": ""Naboo"", ""diameter"": ""12120"", ""rotation_period"": ""26"", ""terrain"": ""grassy hills, swamps, forests, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""312"", ""population"": ""4500000000"", ""url"": ""https://www.swapi.tech/api/planets/8""}"


In [11]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze",
}

# 2 Overwrite

In [12]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return daft.from_pylist(results)

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.with_column("properties", get_properties(daft.col("url")))
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"


bronze_instance = StarWarsBronze(**options)

In [13]:
# Run one table
bronze_instance.load().transform().write(mode="overwrite").execute("people")

2025-03-15 22:16:44 | people | execute | Started
2025-03-15 22:16:44 | people | load | Started
2025-03-15 22:16:49 | people | load | Completed in 0.07 min
2025-03-15 22:16:49 | people | transform | Started
2025-03-15 22:16:49 | people | transform | Completed in 0.0 min
2025-03-15 22:16:49 | people | write | Started


                                                           d

2025-03-15 22:17:26 | people | write | Completed in 0.62 min
2025-03-15 22:17:26 | people | execute | Completed in 0.7 min


In [14]:
# run multiple tables
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-03-15 22:17:26 | people | execute | Started
2025-03-15 22:17:26 | people | load | Started
2025-03-15 22:17:31 | people | load | Completed in 0.08 min
2025-03-15 22:17:31 | people | transform | Started
2025-03-15 22:17:31 | people | transform | Completed in 0.0 min
2025-03-15 22:17:31 | people | write | Started


                                                           d

2025-03-15 22:18:02 | people | write | Completed in 0.5 min
2025-03-15 22:18:02 | people | execute | Completed in 0.6 min
2025-03-15 22:18:02 | planets | execute | Started
2025-03-15 22:18:02 | planets | load | Started


2025-03-15 22:18:05 | planets | load | Completed in 0.05 min
2025-03-15 22:18:05 | planets | transform | Started
2025-03-15 22:18:05 | planets | transform | Completed in 0.0 min
2025-03-15 22:18:05 | planets | write | Started


                                                           d

2025-03-15 22:18:30 | planets | write | Completed in 0.4 min
2025-03-15 22:18:30 | planets | execute | Completed in 0.45 min


In [15]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:17:31.648835,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:17:31.648835,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-15 22:17:31.648835,R2-D2,3,https://www.swapi.tech/api/people/3,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R2-D2"", ""gender"": ""n/a"", ""skin_color"": ""white, blue"", ""hair_color"": ""n/a"", ""height"": ""96"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""33BBY"", ""url"": ""https://www.swapi.tech/api/people/3""}"
2025-03-15 22:17:31.648835,Darth Vader,4,https://www.swapi.tech/api/people/4,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Darth Vader"", ""gender"": ""male"", ""skin_color"": ""white"", ""hair_color"": ""none"", ""height"": ""202"", ""eye_color"": ""yellow"", ""mass"": ""136"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/4""}"
2025-03-15 22:17:31.648835,Leia Organa,5,https://www.swapi.tech/api/people/5,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Leia Organa"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""150"", ""eye_color"": ""brown"", ""mass"": ""49"", ""homeworld"": ""https://www.swapi.tech/api/planets/2"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/5""}"
2025-03-15 22:17:31.648835,Owen Lars,6,https://www.swapi.tech/api/people/6,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Owen Lars"", ""gender"": ""male"", ""skin_color"": ""light"", ""hair_color"": ""brown, grey"", ""height"": ""178"", ""eye_color"": ""blue"", ""mass"": ""120"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""52BBY"", ""url"": ""https://www.swapi.tech/api/people/6""}"
2025-03-15 22:17:31.648835,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Beru Whitesun lars"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""165"", ""eye_color"": ""blue"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""47BBY"", ""url"": ""https://www.swapi.tech/api/people/7""}"
2025-03-15 22:17:31.648835,R5-D4,8,https://www.swapi.tech/api/people/8,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R5-D4"", ""gender"": ""n/a"", ""skin_color"": ""white, red"", ""hair_color"": ""n/a"", ""height"": ""97"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/8""}"


No. Rows: 82


In [16]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/planets")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:18:05.615351,Tatooine,1,https://www.swapi.tech/api/planets/1,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""arid"", ""surface_water"": ""1"", ""name"": ""Tatooine"", ""diameter"": ""10465"", ""rotation_period"": ""23"", ""terrain"": ""desert"", ""gravity"": ""1 standard"", ""orbital_period"": ""304"", ""population"": ""200000"", ""url"": ""https://www.swapi.tech/api/planets/1""}"
2025-03-15 22:18:05.615351,Alderaan,2,https://www.swapi.tech/api/planets/2,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""40"", ""name"": ""Alderaan"", ""diameter"": ""12500"", ""rotation_period"": ""24"", ""terrain"": ""grasslands, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""364"", ""population"": ""2000000000"", ""url"": ""https://www.swapi.tech/api/planets/2""}"
2025-03-15 22:18:05.615351,Yavin IV,3,https://www.swapi.tech/api/planets/3,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate, tropical"", ""surface_water"": ""8"", ""name"": ""Yavin IV"", ""diameter"": ""10200"", ""rotation_period"": ""24"", ""terrain"": ""jungle, rainforests"", ""gravity"": ""1 standard"", ""orbital_period"": ""4818"", ""population"": ""1000"", ""url"": ""https://www.swapi.tech/api/planets/3""}"
2025-03-15 22:18:05.615351,Hoth,4,https://www.swapi.tech/api/planets/4,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""frozen"", ""surface_water"": ""100"", ""name"": ""Hoth"", ""diameter"": ""7200"", ""rotation_period"": ""23"", ""terrain"": ""tundra, ice caves, mountain ranges"", ""gravity"": ""1.1 standard"", ""orbital_period"": ""549"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/4""}"
2025-03-15 22:18:05.615351,Dagobah,5,https://www.swapi.tech/api/planets/5,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""murky"", ""surface_water"": ""8"", ""name"": ""Dagobah"", ""diameter"": ""8900"", ""rotation_period"": ""23"", ""terrain"": ""swamp, jungles"", ""gravity"": ""N/A"", ""orbital_period"": ""341"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/5""}"
2025-03-15 22:18:05.615351,Bespin,6,https://www.swapi.tech/api/planets/6,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""0"", ""name"": ""Bespin"", ""diameter"": ""118000"", ""rotation_period"": ""12"", ""terrain"": ""gas giant"", ""gravity"": ""1.5 (surface), 1 standard (Cloud City)"", ""orbital_period"": ""5110"", ""population"": ""6000000"", ""url"": ""https://www.swapi.tech/api/planets/6""}"
2025-03-15 22:18:05.615351,Endor,7,https://www.swapi.tech/api/planets/7,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""8"", ""name"": ""Endor"", ""diameter"": ""4900"", ""rotation_period"": ""18"", ""terrain"": ""forests, mountains, lakes"", ""gravity"": ""0.85 standard"", ""orbital_period"": ""402"", ""population"": ""30000000"", ""url"": ""https://www.swapi.tech/api/planets/7""}"
2025-03-15 22:18:05.615351,Naboo,8,https://www.swapi.tech/api/planets/8,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""12"", ""name"": ""Naboo"", ""diameter"": ""12120"", ""rotation_period"": ""26"", ""terrain"": ""grassy hills, swamps, forests, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""312"", ""population"": ""4500000000"", ""url"": ""https://www.swapi.tech/api/planets/8""}"


No. Rows: 60


In [17]:
bronze_instance.data["people"].show()

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:17:31.648835,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:17:31.648835,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-15 22:17:31.648835,R2-D2,3,https://www.swapi.tech/api/people/3,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R2-D2"", ""gender"": ""n/a"", ""skin_color"": ""white, blue"", ""hair_color"": ""n/a"", ""height"": ""96"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""33BBY"", ""url"": ""https://www.swapi.tech/api/people/3""}"
2025-03-15 22:17:31.648835,Darth Vader,4,https://www.swapi.tech/api/people/4,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Darth Vader"", ""gender"": ""male"", ""skin_color"": ""white"", ""hair_color"": ""none"", ""height"": ""202"", ""eye_color"": ""yellow"", ""mass"": ""136"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/4""}"
2025-03-15 22:17:31.648835,Leia Organa,5,https://www.swapi.tech/api/people/5,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Leia Organa"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""150"", ""eye_color"": ""brown"", ""mass"": ""49"", ""homeworld"": ""https://www.swapi.tech/api/planets/2"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/5""}"
2025-03-15 22:17:31.648835,Owen Lars,6,https://www.swapi.tech/api/people/6,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Owen Lars"", ""gender"": ""male"", ""skin_color"": ""light"", ""hair_color"": ""brown, grey"", ""height"": ""178"", ""eye_color"": ""blue"", ""mass"": ""120"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""52BBY"", ""url"": ""https://www.swapi.tech/api/people/6""}"
2025-03-15 22:17:31.648835,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Beru Whitesun lars"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""165"", ""eye_color"": ""blue"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""47BBY"", ""url"": ""https://www.swapi.tech/api/people/7""}"
2025-03-15 22:17:31.648835,R5-D4,8,https://www.swapi.tech/api/people/8,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R5-D4"", ""gender"": ""n/a"", ""skin_color"": ""white, red"", ""hair_color"": ""n/a"", ""height"": ""97"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/8""}"


In [18]:
bronze_instance.data["planets"].show()

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:18:05.615351,Tatooine,1,https://www.swapi.tech/api/planets/1,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""arid"", ""surface_water"": ""1"", ""name"": ""Tatooine"", ""diameter"": ""10465"", ""rotation_period"": ""23"", ""terrain"": ""desert"", ""gravity"": ""1 standard"", ""orbital_period"": ""304"", ""population"": ""200000"", ""url"": ""https://www.swapi.tech/api/planets/1""}"
2025-03-15 22:18:05.615351,Alderaan,2,https://www.swapi.tech/api/planets/2,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""40"", ""name"": ""Alderaan"", ""diameter"": ""12500"", ""rotation_period"": ""24"", ""terrain"": ""grasslands, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""364"", ""population"": ""2000000000"", ""url"": ""https://www.swapi.tech/api/planets/2""}"
2025-03-15 22:18:05.615351,Yavin IV,3,https://www.swapi.tech/api/planets/3,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate, tropical"", ""surface_water"": ""8"", ""name"": ""Yavin IV"", ""diameter"": ""10200"", ""rotation_period"": ""24"", ""terrain"": ""jungle, rainforests"", ""gravity"": ""1 standard"", ""orbital_period"": ""4818"", ""population"": ""1000"", ""url"": ""https://www.swapi.tech/api/planets/3""}"
2025-03-15 22:18:05.615351,Hoth,4,https://www.swapi.tech/api/planets/4,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""frozen"", ""surface_water"": ""100"", ""name"": ""Hoth"", ""diameter"": ""7200"", ""rotation_period"": ""23"", ""terrain"": ""tundra, ice caves, mountain ranges"", ""gravity"": ""1.1 standard"", ""orbital_period"": ""549"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/4""}"
2025-03-15 22:18:05.615351,Dagobah,5,https://www.swapi.tech/api/planets/5,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""murky"", ""surface_water"": ""8"", ""name"": ""Dagobah"", ""diameter"": ""8900"", ""rotation_period"": ""23"", ""terrain"": ""swamp, jungles"", ""gravity"": ""N/A"", ""orbital_period"": ""341"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/5""}"
2025-03-15 22:18:05.615351,Bespin,6,https://www.swapi.tech/api/planets/6,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""0"", ""name"": ""Bespin"", ""diameter"": ""118000"", ""rotation_period"": ""12"", ""terrain"": ""gas giant"", ""gravity"": ""1.5 (surface), 1 standard (Cloud City)"", ""orbital_period"": ""5110"", ""population"": ""6000000"", ""url"": ""https://www.swapi.tech/api/planets/6""}"
2025-03-15 22:18:05.615351,Endor,7,https://www.swapi.tech/api/planets/7,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""8"", ""name"": ""Endor"", ""diameter"": ""4900"", ""rotation_period"": ""18"", ""terrain"": ""forests, mountains, lakes"", ""gravity"": ""0.85 standard"", ""orbital_period"": ""402"", ""population"": ""30000000"", ""url"": ""https://www.swapi.tech/api/planets/7""}"
2025-03-15 22:18:05.615351,Naboo,8,https://www.swapi.tech/api/planets/8,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""12"", ""name"": ""Naboo"", ""diameter"": ""12120"", ""rotation_period"": ""26"", ""terrain"": ""grassy hills, swamps, forests, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""312"", ""population"": ""4500000000"", ""url"": ""https://www.swapi.tech/api/planets/8""}"


# 3 Replace Where

In [19]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return daft.from_pylist(results)

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.with_column("properties", get_properties(daft.col("url")))
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"
    
    def get_replace_condition(self, df: daft.DataFrame, table: str) -> str:
        return "uid > '0'"


bronze_instance = StarWarsBronze(**options)
bronze_instance.load().transform().write(mode="replace").execute("people")

2025-03-15 22:18:36 | people | execute | Started
2025-03-15 22:18:36 | people | load | Started
2025-03-15 22:18:41 | people | load | Completed in 0.07 min
2025-03-15 22:18:41 | people | transform | Started
2025-03-15 22:18:41 | people | transform | Completed in 0.0 min
2025-03-15 22:18:41 | people | write | Started


                                                      d

2025-03-15 22:19:14 | people | write | Completed in 0.53 min


2025-03-15 22:19:14 | people | execute | Completed in 0.62 min


In [20]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:18:41.556405,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:18:41.556405,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-15 22:18:41.556405,R2-D2,3,https://www.swapi.tech/api/people/3,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R2-D2"", ""gender"": ""n/a"", ""skin_color"": ""white, blue"", ""hair_color"": ""n/a"", ""height"": ""96"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""33BBY"", ""url"": ""https://www.swapi.tech/api/people/3""}"
2025-03-15 22:18:41.556405,Darth Vader,4,https://www.swapi.tech/api/people/4,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Darth Vader"", ""gender"": ""male"", ""skin_color"": ""white"", ""hair_color"": ""none"", ""height"": ""202"", ""eye_color"": ""yellow"", ""mass"": ""136"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/4""}"
2025-03-15 22:18:41.556405,Leia Organa,5,https://www.swapi.tech/api/people/5,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Leia Organa"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""150"", ""eye_color"": ""brown"", ""mass"": ""49"", ""homeworld"": ""https://www.swapi.tech/api/planets/2"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/5""}"
2025-03-15 22:18:41.556405,Owen Lars,6,https://www.swapi.tech/api/people/6,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Owen Lars"", ""gender"": ""male"", ""skin_color"": ""light"", ""hair_color"": ""brown, grey"", ""height"": ""178"", ""eye_color"": ""blue"", ""mass"": ""120"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""52BBY"", ""url"": ""https://www.swapi.tech/api/people/6""}"
2025-03-15 22:18:41.556405,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Beru Whitesun lars"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""165"", ""eye_color"": ""blue"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""47BBY"", ""url"": ""https://www.swapi.tech/api/people/7""}"
2025-03-15 22:18:41.556405,R5-D4,8,https://www.swapi.tech/api/people/8,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R5-D4"", ""gender"": ""n/a"", ""skin_color"": ""white, red"", ""hair_color"": ""n/a"", ""height"": ""97"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/8""}"


No. Rows: 82


# 4 Append

In [21]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return daft.from_pylist(results)

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.with_column("properties", get_properties(daft.col("url")))
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"


bronze_instance = StarWarsBronze(**options)
bronze_instance.load().transform().write().execute("people")  # default mode is append

2025-03-15 22:19:14 | people | execute | Started
2025-03-15 22:19:14 | people | load | Started
2025-03-15 22:19:19 | people | load | Completed in 0.07 min
2025-03-15 22:19:19 | people | transform | Started
2025-03-15 22:19:19 | people | transform | Completed in 0.0 min
2025-03-15 22:19:19 | people | write | Started


                                                           d

2025-03-15 22:19:54 | people | write | Completed in 0.57 min
2025-03-15 22:19:54 | people | execute | Completed in 0.65 min


In [22]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:19:19.282962,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:19:19.282962,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-15 22:19:19.282962,R2-D2,3,https://www.swapi.tech/api/people/3,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R2-D2"", ""gender"": ""n/a"", ""skin_color"": ""white, blue"", ""hair_color"": ""n/a"", ""height"": ""96"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""33BBY"", ""url"": ""https://www.swapi.tech/api/people/3""}"
2025-03-15 22:19:19.282962,Darth Vader,4,https://www.swapi.tech/api/people/4,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Darth Vader"", ""gender"": ""male"", ""skin_color"": ""white"", ""hair_color"": ""none"", ""height"": ""202"", ""eye_color"": ""yellow"", ""mass"": ""136"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/4""}"
2025-03-15 22:19:19.282962,Leia Organa,5,https://www.swapi.tech/api/people/5,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Leia Organa"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""150"", ""eye_color"": ""brown"", ""mass"": ""49"", ""homeworld"": ""https://www.swapi.tech/api/planets/2"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/5""}"
2025-03-15 22:19:19.282962,Owen Lars,6,https://www.swapi.tech/api/people/6,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Owen Lars"", ""gender"": ""male"", ""skin_color"": ""light"", ""hair_color"": ""brown, grey"", ""height"": ""178"", ""eye_color"": ""blue"", ""mass"": ""120"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""52BBY"", ""url"": ""https://www.swapi.tech/api/people/6""}"
2025-03-15 22:19:19.282962,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Beru Whitesun lars"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""165"", ""eye_color"": ""blue"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""47BBY"", ""url"": ""https://www.swapi.tech/api/people/7""}"
2025-03-15 22:19:19.282962,R5-D4,8,https://www.swapi.tech/api/people/8,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R5-D4"", ""gender"": ""n/a"", ""skin_color"": ""white, red"", ""hair_color"": ""n/a"", ""height"": ""97"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/8""}"


No. Rows: 164


# 5 Merge

In [23]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return daft.from_pylist(results)

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.with_column("properties", get_properties(daft.col("url")))
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"
    
    def get_delta_merge_builder(
        self, df: daft.DataFrame, delta_table: DeltaTable
    ) -> TableMerger:
        merge_condition = "target.uid = source.uid"
        builder = delta_table.merge(df, predicate=merge_condition, source_alias="source", target_alias="target")
        builder = builder.when_matched_update_all()
        builder = builder.when_not_matched_insert_all()
        return builder


bronze_instance = StarWarsBronze(**options)
bronze_instance.load().transform().write(mode="merge").execute("people")

2025-03-15 22:19:54 | people | execute | Started
2025-03-15 22:19:54 | people | load | Started
2025-03-15 22:19:59 | people | load | Completed in 0.08 min
2025-03-15 22:19:59 | people | transform | Started
2025-03-15 22:19:59 | people | transform | Completed in 0.0 min
2025-03-15 22:19:59 | people | write | Started


                                                      d

2025-03-15 22:20:33 | people | write | Completed in 0.57 min
2025-03-15 22:20:33 | people | execute | Completed in 0.65 min


In [24]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:19:59.723442,Greedo,15,https://www.swapi.tech/api/people/15,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Greedo"", ""gender"": ""male"", ""skin_color"": ""green"", ""hair_color"": ""n/a"", ""height"": ""173"", ""eye_color"": ""black"", ""mass"": ""74"", ""homeworld"": ""https://www.swapi.tech/api/planets/23"", ""birth_year"": ""44BBY"", ""url"": ""https://www.swapi.tech/api/people/15""}"
2025-03-15 22:19:59.723442,Nute Gunray,33,https://www.swapi.tech/api/people/33,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Nute Gunray"", ""gender"": ""male"", ""skin_color"": ""mottled green"", ""hair_color"": ""none"", ""height"": ""191"", ""eye_color"": ""red"", ""mass"": ""90"", ""homeworld"": ""https://www.swapi.tech/api/planets/18"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/33""}"
2025-03-15 22:19:59.723442,Rugor Nass,38,https://www.swapi.tech/api/people/38,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Rugor Nass"", ""gender"": ""male"", ""skin_color"": ""green"", ""hair_color"": ""none"", ""height"": ""206"", ""eye_color"": ""orange"", ""mass"": ""unknown"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/38""}"
2025-03-15 22:19:59.723442,Kit Fisto,53,https://www.swapi.tech/api/people/53,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Kit Fisto"", ""gender"": ""male"", ""skin_color"": ""green"", ""hair_color"": ""none"", ""height"": ""196"", ""eye_color"": ""black"", ""mass"": ""87"", ""homeworld"": ""https://www.swapi.tech/api/planets/44"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/53""}"
2025-03-15 22:19:59.723442,Leia Organa,5,https://www.swapi.tech/api/people/5,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Leia Organa"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""150"", ""eye_color"": ""brown"", ""mass"": ""49"", ""homeworld"": ""https://www.swapi.tech/api/planets/2"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/5""}"
2025-03-15 22:19:59.723442,Sly Moore,82,https://www.swapi.tech/api/people/82,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Sly Moore"", ""gender"": ""female"", ""skin_color"": ""pale"", ""hair_color"": ""none"", ""height"": ""178"", ""eye_color"": ""white"", ""mass"": ""48"", ""homeworld"": ""https://www.swapi.tech/api/planets/60"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/82""}"
2025-03-15 22:19:59.723442,Palpatine,21,https://www.swapi.tech/api/people/21,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Palpatine"", ""gender"": ""male"", ""skin_color"": ""pale"", ""hair_color"": ""grey"", ""height"": ""170"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""82BBY"", ""url"": ""https://www.swapi.tech/api/people/21""}"
2025-03-15 22:19:59.723442,Roos Tarpals,37,https://www.swapi.tech/api/people/37,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Roos Tarpals"", ""gender"": ""male"", ""skin_color"": ""grey"", ""hair_color"": ""none"", ""height"": ""224"", ""eye_color"": ""orange"", ""mass"": ""82"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/37""}"


No. Rows: 164


# 6 Clean Up

In [25]:
import shutil
shutil.rmtree(f"D:/Data/{CATALOG}")